# False Positive Analysis of Patent Novelty Classification Models

In machine learning classification tasks, evaluating model performance using accuracy alone is often insufficient. Understanding **how and why models make incorrect predictions** provides deeper insights into model behavior and helps identify potential improvements.

One important type of error is the **False Positive**. A false positive occurs when a model incorrectly predicts a class label for a sample that actually belongs to another class.

In this analysis, false positives are examined for multiple machine learning models trained to classify **patents into novelty tiers** based on their textual content. The goal is to identify **patterns in misclassified patents** and better understand the limitations of the models.

Purpose of False Positive Analysis

Analyzing false positives helps in:

* Identifying **patterns in misclassified patents**
* Understanding which **types of patent descriptions confuse the models**
* Detecting **overlapping linguistic patterns between novelty tiers**
* Improving feature engineering and model design

This step is crucial for improving the **reliability and interpretability** of the patent novelty classification system.

---



### Step 1 – Loading Libraries and Dataset

In this step, the necessary Python libraries required for data processing and model evaluation are imported. These libraries provide functionality for handling datasets, performing machine learning operations, and analyzing model predictions.

After importing the required libraries, the **balanced patent dataset** created during the preprocessing stage is loaded into the environment.

Loading this dataset ensures that the same processed data used during the model development stage is available for performing further evaluation and **false positive analysis**.


In [2]:
import pandas as pd
import joblib
from collections import Counter
import os

In [ ]:

df = pd.read_csv("final_patent_dataset_balanced.csv")

print("Dataset Loaded:", df.shape)

df.head()

Dataset Loaded: (5000, 2)


,clean_text,novelty tier
0,power generation unit oceanographic sensor moo...,0
1,novel container design built overrun meter pre...,2
2,system rapid bake semiconductor substrate uppe...,2
3,electromechanical brake booster disclosed elec...,1
4,time pressure dosing fuel additive delivery sy...,1


In [4]:
X_text = df["clean_text"]
y = df["novelty tier"]

### Step 2 - Train–Test Split

To evaluate the models on unseen data, the dataset is divided into:

* **Training Set (80%)**
* **Testing Set (20%)**

Stratified sampling is used to ensure that the **distribution of novelty tiers remains consistent** across both sets. This helps maintain balanced evaluation conditions.

The testing set is later used to analyze **false positive predictions** produced by the trained models.


In [5]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Test Samples:", len(X_test_text))

Test Samples: 1000


### Step 3 - Loading Preprocessing Objects

During training, the patent text was converted into numerical features using two preprocessing techniques:

- **TF-IDF Vectorization** - TF-IDF (Term Frequency–Inverse Document Frequency) converts textual data into numerical vectors by measuring the importance of words within the patent corpus.

- **Chi-Square Feature Selection** - Chi-square feature selection reduces dimensionality by selecting the **most statistically significant words** associated with the novelty tiers.

Instead of retraining these objects, the saved versions are loaded using **Joblib** to ensure the test data is processed in the same way as the training data.


In [6]:
vectorizer = joblib.load(r"C:\Users\jewel\saved_models\tfidf_vectorizer.pkl")
selector = joblib.load(r"C:\Users\jewel\saved_models\chi2_selector.pkl")

print("Preprocessing objects loaded successfully.")

Preprocessing objects loaded successfully.


### Step 4 - Text Feature Transformation

After loading the preprocessing objects, the testing text data undergoes two transformations:

1. **TF-IDF Transformation**: Converts patent text into a high-dimensional feature vector.

2. **Feature Selection (Chi-Square)**: Filters the TF-IDF features to retain only the most relevant ones.

This results in a **feature matrix** that is compatible with the trained machine learning models.


In [ ]:

X_test = vectorizer.transform(X_test_text)
X_test = selector.transform(X_test)

print("Feature matrix shape:", X_test.shape)

Feature matrix shape: (1000, 8000)


### Step 5 - Loading Trained Machine Learning Models

Several trained models are loaded to perform predictions on the testing dataset. These models were previously trained and saved during the model development stage.

The models include:

* Multinomial Naive Bayes
* Logistic Regression
* Support Vector Machine
* Random Forest
* Gradient Boosting
* XGBoost

Each model represents a different approach to classification, allowing comparison of how each algorithm handles patent novelty prediction.


In [8]:
models = {
    "Multinomial Naive Bayes": joblib.load(r"C:\Users\jewel\saved_models\Multinomial_Naive_Bayes_model.pkl"),
    "Logistic Regression": joblib.load(r"C:\Users\jewel\saved_models\Logistic_Regression_model.pkl"),
    "Support Vector Machine": joblib.load(r"C:\Users\jewel\saved_models\Support_Vector_Machine_model.pkl"),
    "Random Forest": joblib.load(r"C:\Users\jewel\saved_models\Random_Forest_model.pkl"),
    "Gradient Boosting": joblib.load(r"C:\Users\jewel\saved_models\Gradient_Boosting_model.pkl"),
    "XGBoost": joblib.load(r"C:\Users\jewel\saved_models\XGBoost_model.pkl")
}

print("All models loaded successfully.")

All models loaded successfully.


### Step 6 - Performing False Positive Analysis

For each model, predictions are generated using the processed test dataset.

The following steps are performed:

1. The model predicts the **novelty tier** for each patent.
2. A results dataframe is created containing:

   * Patent text
   * True label
   * Predicted label
3. Predictions are compared with the true labels to identify **incorrect classifications**.

The analysis specifically focuses on **false positives**, where the model predicts a novelty tier incorrectly.



In [ ]:
from collections import Counter
import pandas as pd

false_positive_summary = []
top_words_all_models = []

for name, model in models.items():

    y_pred = model.predict(X_test)

    results = pd.DataFrame({
        "text": X_test_text.values,
        "true_label": y_test.values,
        "predicted_label": y_pred
    })

    false_positives = results[
        (results["true_label"] == 0) &
        (results["predicted_label"] == 3)
    ]
    fp_count = len(false_positives)

    false_positive_summary.append({
        "Model": name,
        "False Positives": fp_count
    })

    words = " ".join(false_positives["text"].astype(str)).split()
    common_words = Counter(words).most_common(20)

    words_df = pd.DataFrame(common_words, columns=["Word", "Frequency"])
    words_df["Model"] = name

    top_words_all_models.append(words_df)
    filename = f"false_positive_examples_{name.replace(' ','_')}.csv"
    false_positives.to_csv(filename, index=False)



In [11]:
false_positive_summary_df = pd.DataFrame(false_positive_summary)

top_words_df = pd.concat(top_words_all_models, ignore_index=True)

print("\nFalse Positive Summary:")
print(false_positive_summary_df)

print("\nTop Words in False Positives:")
print(top_words_df.head(50))



False Positive Summary:
                     Model  False Positives
0  Multinomial Naive Bayes                4
1      Logistic Regression                2
2   Support Vector Machine                0
3            Random Forest                0
4        Gradient Boosting                0
5                  XGBoost                0

Top Words in False Positives:
              Word Frequency                    Model
0          picture       174  Multinomial Naive Bayes
1            first        85  Multinomial Naive Bayes
2           second        69  Multinomial Naive Bayes
3   authentication        67  Multinomial Naive Bayes
4        specified        64  Multinomial Naive Bayes
5      information        62  Multinomial Naive Bayes
6            using        47  Multinomial Naive Bayes
7          decoded        46  Multinomial Naive Bayes
8       identifier        45  Multinomial Naive Bayes
9            intra        42  Multinomial Naive Bayes
10            data        41  Multinomial 

# Interpretation of False Positive Analysis

The false positive analysis reveals that **very few misclassifications occurred across the evaluated models**, indicating that the trained models are generally effective at distinguishing between the patent novelty tiers.

From the summary results, **Multinomial Naive Bayes produced the highest number of false positives (4 cases)**, followed by **Logistic Regression with 2 cases**. In contrast, **Support Vector Machine, Random Forest, Gradient Boosting, and XGBoost produced zero false positives** for the examined condition (true label = 0 predicted as label = 3). This suggests that the more advanced models were able to better capture complex patterns within the patent text and avoid incorrectly classifying lower novelty patents as highly novel ones.

The word frequency analysis of the false positive samples highlights several frequently occurring terms such as **“picture”, “authentication”, “identifier”, “image”, “vector”, “decoded”, “data”, and “block”**. These terms are commonly associated with **image processing, multimedia systems, authentication mechanisms, and data processing technologies**. Because such technical vocabulary often appears in patents describing advanced computational methods, simpler models like **Naive Bayes and Logistic Regression may interpret these words as strong indicators of higher novelty**, even when the patent actually belongs to a lower novelty tier.

Additionally, words like **“photograph”, “face”, “encrypted”, “key”, and “database”** appearing in the Logistic Regression false positives suggest that patents related to **security, biometric identification, and data management** may contain terminology that overlaps with high-novelty patents. This overlap can lead to incorrect predictions when the model relies heavily on word frequency patterns.

Overall, the results indicate that **probabilistic and linear models are more prone to false positive errors**, while **margin-based and ensemble models (SVM, Random Forest, Gradient Boosting, and XGBoost) demonstrate stronger discrimination ability** for patent novelty classification. This highlights the advantage of more sophisticated models in handling complex textual patterns within technical patent documents.
